In [ ]:
# 放在文件最顶部：在任何 sklearn / optuna 导入之前
import os
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*sklearn\.utils\.parallel\.delayed.*Parallel.*",
    category=UserWarning
)







# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

# =========================
# 0) 配置 & 尽量屏蔽警告
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# 直接读取你已生成的 Morgan 特征文件（xlsx）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

# RandomForest 固定参数（Optuna 只搜索部分超参）
BASE_RF_PARAMS = dict(
    n_jobs=-1,
    random_state=RANDOM_SEED,
)

# =========================
# 1) 读取 transformed Morgan 特征文件 & 自动识别列
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到 Morgan 特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols_morgan(df: pd.DataFrame):
    """
    兼容两种常见 Morgan 保存方式：
    A) 列名是 morgan_0...morgan_2047 / fp_ / bit_
    B) 列名是纯数字 0...2047（Excel 读入后可能变成 int）
    """
    # A) 前缀形式
    morgan_cols = []
    for prefix in ["Rule__","FG__","morgan_", "fp_", "ecfp_", "mfp_", "bit_","KG_emb_"]:
        morgan_cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])
    if morgan_cols:
        return list(morgan_cols)

    # B) 纯数字列
    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)

    if num_cols:
        def keyf(x):
            try:
                return int(x)
            except Exception:
                return 10**18
        return sorted(num_cols, key=keyf)

    raise ValueError(
        "未找到 Morgan 特征列。\n"
        "请确保：Morgan 列名为 morgan_/fp_/bit_ 前缀，或列名为 0..2047 的纯数字。"
    )

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中仍包含 138 个标签列。")
    return label_cols

def load_Xy_from_transformed_morgan(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    smiles_col = find_smiles_col(df)
    feature_cols = infer_feature_cols_morgan(df)
    label_cols = infer_label_cols(df, smiles_col, feature_cols)

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols, df

# =========================
# 2) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int, classes_list=None):
    """
    兼容：
    - multi-output RF 的 predict_proba: list，长度=labels，每个 (n, n_classes_k)
    - 处理某标签在该 fold 训练集里只有单类 => (n,1)
    """
    if isinstance(p, list):
        out = np.zeros((p[0].shape[0], n_labels), dtype=np.float32)
        for k in range(n_labels):
            pk = p[k]
            if pk.ndim != 2:
                raise ValueError(f"predict_proba[{k}] 形状异常: {pk.shape}")

            if pk.shape[1] == 2:
                # 取 class==1 的那一列更稳（避免 classes_ 顺序不是 [0,1]）
                if classes_list is not None and len(classes_list) == n_labels:
                    cls = list(classes_list[k])
                    if 1 in cls:
                        j = cls.index(1)
                        out[:, k] = pk[:, j].astype(np.float32)
                    else:
                        out[:, k] = 0.0
                else:
                    out[:, k] = pk[:, 1].astype(np.float32)

            elif pk.shape[1] == 1:
                # 只有单一类别：若该类别是 1 => 概率恒为1；若是 0 => 概率恒为0
                if classes_list is not None and len(classes_list) == n_labels:
                    only_cls = int(list(classes_list[k])[0])
                    out[:, k] = 1.0 if only_cls == 1 else 0.0
                else:
                    # 兜底：默认认为是 class 0
                    out[:, k] = 0.0
            else:
                raise ValueError(f"predict_proba[{k}] 类别数异常: {pk.shape[1]}")
        return out

    # 非 list 情况（一般不会出现在 multi-output RF）
    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 3) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 4) 5 折 CV 评估（RF）
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = RandomForestClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)  # list(length=L)
        y_prob = extract_positive_proba(p, n_labels=n_labels, classes_list=getattr(clf, "classes_", None))

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]
    return mean_metrics

# =========================
# 5) Optuna 超参空间（RF）
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_RF_PARAMS)

    max_depth_raw = trial.suggest_int("max_depth", 0, 40)  # 0 表示 None
    max_depth = None if max_depth_raw == 0 else max_depth_raw

    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": max_depth,
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", 0.3, 0.5, 0.8, 1.0]),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        "criterion": trial.suggest_categorical("criterion", ["gini", "entropy", "log_loss"]),
    })
    return params

# =========================
# 6) 主流程：读Morgan特征 -> 固定folds -> Optuna(30) -> 输出+保存
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed Morgan feature file:", feature_file)

    X, y, feat_cols, label_cols, df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)
        mean_metrics = cv_eval_one_paramset(X, y, folds, params, thresh=THRESH)

        score = mean_metrics["AUPRC_macro"]  # PRAUC
        print(
            f"\n[TRIAL {trial.number:02d}] score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "AUPRC_macro": t.value, **t.params}
        row.update({k: v for k, v in t.user_attrs.items() if k.endswith("_macro")})
        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values("AUPRC_macro", ascending=False)
    out_csv = "./optuna_singlemodel_RF_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

if __name__ == "__main__":
    main()


[INFO] Using transformed Morgan feature file: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx


[I 2026-01-29 23:08:51,237] A new study created in memory with name: no-name-5d5d805f-ad18-49d7-9ead-255a8d11ccd5


[INFO] X shape=(4952, 2627) (features=2627) | y shape=(4952, 138) (labels=138)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] score(AUPRC_macro)=0.325311 | AUROC=0.875800 | Acc=0.972269 | P=0.471021 | R=0.217710 | Spec=0.991173
[I 2026-01-30 05:51:02,292] Trial 0 finished with value: 0.3253107636382795 and parameters: {'n_estimators': 637, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 0.7978542347074177, 'reg_lambda': 0.0017775399007348214, 'reg_alpha': 0.3426417745118369, 'gamma': 3.005575058716044}. Best is trial 0 with value: 0.3253107636382795.

[TRIAL 01] score(AUPRC_macro)=0.323472 | AUROC=0.876430 | Acc=0.972463 | P=0.464144 | R=0.215321 | Spec=0.991610
[I 2026-01-30 07:28:59,249] Trial 1 finished with value: 0.323472479554622 and parameters: {'n_estimators': 937, 'max_depth': 3, 'learning_rate': 0.18276027831785724, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'min_child_weight': 0.8620446097910766, 'reg_lambda': 0.006149337057087106, 'reg_alpha': 4.431942789151

In [1]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import warnings
import numpy as np
import pandas as pd

import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.multioutput import MultiOutputClassifier

# =========================
# 0) 全局：尽量屏蔽警告 + LightGBM 日志
# =========================
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    import lightgbm as lgb
except Exception as e:
    raise RuntimeError(
        "未检测到 lightgbm。请先安装：pip install lightgbm\n"
        f"原始错误：{repr(e)}"
    )

# =========================
# 1) 配置
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的 Morgan 特征文件（xlsx）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"

# 固定参数（Optuna 只搜索部分超参）
# 关键：verbosity=-1 关闭 LightGBM 输出
BASE_LGB_PARAMS = dict(
    objective="binary",
    boosting_type="gbdt",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=-1,  # 关闭 LightGBM 日志（重要）
)

# =========================
# 2) 读取 transformed Morgan 特征文件 & 自动识别列
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到 Morgan 特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols_morgan(df: pd.DataFrame):
    """
    兼容两种常见 Morgan 保存方式：
    A) 列名是 morgan_0... / fp_ / bit_
    B) 列名是纯数字 0...2047（Excel 读入后可能变成 int）
    """
    morgan_cols = []
    for prefix in ["Rule__","FG__","morgan_", "fp_", "ecfp_", "mfp_", "bit_","KG_emb_"]:
        morgan_cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])
    if morgan_cols:
        return list(morgan_cols)

    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)

    if num_cols:
        def keyf(x):
            try:
                return int(x)
            except Exception:
                return 10**18
        return sorted(num_cols, key=keyf)

    raise ValueError(
        "未找到 Morgan 特征列。\n"
        "请确保：Morgan 列名为 morgan_/fp_/bit_ 前缀，或列名为 0..2047 的纯数字。"
    )

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中仍包含 138 个标签列。")
    return label_cols

def load_Xy_from_transformed_morgan(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    smiles_col = find_smiles_col(df)
    feature_cols = infer_feature_cols_morgan(df)
    label_cols = infer_label_cols(df, smiles_col, feature_cols)

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    # MultiOutputClassifier.predict_proba -> list，长度=labels，每个是 (n,2)
    if isinstance(p, list):
        return np.vstack([pi[:, 1] for pi in p]).T.astype(np.float32)

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 5 折 CV 评估（LightGBM + MultiOutput）
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        base_est = lgb.LGBMClassifier(**params)
        # 每个标签一个 LGBM；n_jobs=1 避免“外层并行 + 内层线程”导致爆核
        clf = MultiOutputClassifier(base_est, n_jobs=1)

        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]
    return mean_metrics

# =========================
# 6) Optuna 超参空间（LightGBM）
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_LGB_PARAMS)
    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 2000),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),

        "num_leaves": trial.suggest_int("num_leaves", 31, 255),
        "max_depth": trial.suggest_int("max_depth", -1, 20),  # -1 表示不限制
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 80),

        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),

        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 50.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),

        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 5.0),  # 类似 gamma
    })
    return params

# =========================
# 7) 主流程：读Morgan特征 -> 固定folds -> Optuna(30) -> 输出+保存
# =========================
def main():
    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed Morgan feature file:", feature_file)

    X, y, feat_cols, label_cols, df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)
        mean_metrics = cv_eval_one_paramset(X, y, folds, params, thresh=THRESH)

        score = mean_metrics["AUPRC_macro"]  # PRAUC
        print(
            f"\n[TRIAL {trial.number:02d}] score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "AUPRC_macro": t.value, **t.params}
        row.update({k: v for k, v in t.user_attrs.items() if k.endswith("_macro")})
        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values("AUPRC_macro", ascending=False)
    out_csv = "./optuna_singlemodel_LIGHTGBM_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

if __name__ == "__main__":
    main()


[INFO] Using transformed Morgan feature file: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx


[I 2026-01-23 11:52:52,735] A new study created in memory with name: no-name-4d2844de-2f60-41fe-b857-916f311b1c4f


[INFO] X shape=(4952, 2627) (features=2627) | y shape=(4952, 138) (labels=138)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] score(AUPRC_macro)=0.267252 | AUROC=0.853727 | Acc=0.972680 | P=0.734156 | R=0.067979 | Spec=0.996191
[I 2026-01-23 12:03:26,118] Trial 0 finished with value: 0.26725223227075745 and parameters: {'n_estimators': 937, 'learning_rate': 0.17254716573280354, 'num_leaves': 195, 'max_depth': 12, 'min_child_samples': 16, 'subsample': 0.662397808134481, 'colsample_bytree': 0.6232334448672797, 'reg_lambda': 11.752647960576219, 'reg_alpha': 0.002570603566117598, 'min_split_gain': 3.540362888980227}. Best is trial 0 with value: 0.26725223227075745.

[TRIAL 01] score(AUPRC_macro)=0.311260 | AUROC=0.864046 | Acc=0.973088 | P=0.533917 | R=0.152207 | Spec=0.994305
[I 2026-01-23 12:07:49,908] Trial 1 finished with value: 0.3112596947600969 and parameters: {'n_estimators': 335, 'learning_rate': 0.18276027831785724, 'num_leaves': 218, 'max_depth': 3, 'min_child_samples': 18, 'subsample': 0.6733618039413735, 'colsample_bytree': 0.7216968971838151, 'reg_lambda': 0.2922905212920093, 'reg_alpha'

In [ ]:
# -*- coding: utf-8 -*-
import os
import re
import glob
import json
import numpy as np
import pandas as pd

import optuna
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

# =========================
# 0) 配置
# =========================
RANDOM_SEED = 42
N_TRIALS = 30
N_SPLITS = 5
THRESH = 0.5

# 直接读取你已生成的 Morgan 特征文件（xlsx）
FEATURE_FILE_GLOB = "./Malodors_Rule&FG&Morgan&StructKG_features.xlsx"
# 如果你的文件名不同，改成你的，例如：
# FEATURE_FILE_GLOB = "./Malodors_transformed_MORGAN_features.xlsx"

# 固定参数（Optuna 只搜索部分超参）
BASE_XGB_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_SEED,
    verbosity=0,
    multi_strategy="multi_output_tree",
)


# =========================
# 1) 版本检查
# =========================
def _ver_tuple(v: str):
    parts = re.split(r"[.+-]", v.strip())
    nums = []
    for p in parts[:3]:
        try:
            nums.append(int(p))
        except Exception:
            nums.append(0)
    while len(nums) < 3:
        nums.append(0)
    return tuple(nums)

def check_xgb_version():
    v = _ver_tuple(xgb.__version__)
    if v < (1, 6, 0):
        raise RuntimeError(f"xgboost=={xgb.__version__} 太旧：multi-label 需要 >=1.6")
    if "multi_strategy" in BASE_XGB_PARAMS and v < (2, 0, 0):
        raise RuntimeError(
            f"xgboost=={xgb.__version__} 不支持 multi_output_tree（需要>=2.0）。"
            f"升级或删掉 multi_strategy。"
        )

# =========================
# 2) 读取 transformed Morgan 特征文件 & 自动识别列
# =========================
def find_feature_file(pattern: str):
    cands = sorted(glob.glob(pattern))
    if not cands:
        raise FileNotFoundError(
            f"找不到 Morgan 特征文件：{pattern}\n"
            f"请确认你保存的 xlsx 文件名，或修改 FEATURE_FILE_GLOB。"
        )
    return cands[-1]

def find_smiles_col(df: pd.DataFrame):
    cand = [c for c in df.columns if isinstance(c, str) and "smiles" in c.lower()]
    if not cand:
        return None
    for p in ["Canonical SMILES", "canonical_smiles", "SMILES", "smiles"]:
        for c in cand:
            if c.lower() == p.lower():
                return c
    return cand[0]

def is_binary_01_series(s: pd.Series) -> bool:
    if s.dtype == bool:
        return True
    if not np.issubdtype(s.dtype, np.number):
        return False
    vals = pd.unique(s.dropna())
    if len(vals) == 0:
        return False
    return set(vals).issubset({0, 1})

def infer_feature_cols_morgan(df: pd.DataFrame):
    """
    兼容两种常见 Morgan 保存方式：
    A) 列名是 morgan_0...morgan_2047 / fp_ / bit_
    B) 列名是纯数字 0...2047（Excel 读入后可能变成 int）
    """
    # A) 前缀形式
    morgan_cols = []
    for prefix in ["Rule__","FG__","morgan_", "fp_", "ecfp_", "mfp_", "bit_","KG_emb_"]:
        morgan_cols.extend([c for c in df.columns if isinstance(c, str) and c.startswith(prefix)])

    if morgan_cols:
        # 保持原顺序
        return list(morgan_cols)

    # B) 纯数字列
    num_cols = []
    for c in df.columns:
        if isinstance(c, (int, np.integer)):
            num_cols.append(c)
        elif isinstance(c, str) and c.isdigit():
            num_cols.append(c)

    if num_cols:
        # 尽量按数值排序
        def keyf(x):
            try:
                return int(x)
            except Exception:
                return 10**18
        return sorted(num_cols, key=keyf)

    raise ValueError(
        "未找到 Morgan 特征列。\n"
        "请确保：Morgan 列名为 morgan_/fp_/bit_ 前缀，或列名为 0..2047 的纯数字。"
    )

def infer_label_cols(df: pd.DataFrame, smiles_col: str, feature_cols: list):
    exclude = set(feature_cols)
    if smiles_col is not None:
        exclude.add(smiles_col)

    label_cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if is_binary_01_series(df[c]):
            label_cols.append(c)

    if not label_cols:
        raise ValueError("未识别到标签列（0/1）。请确认文件中仍包含 138 个标签列。")

    return label_cols

def load_Xy_from_transformed_morgan(xlsx_path: str):
    df = pd.read_excel(xlsx_path)

    smiles_col = find_smiles_col(df)
    feature_cols = infer_feature_cols_morgan(df)
    label_cols = infer_label_cols(df, smiles_col, feature_cols)

    X = df[feature_cols].fillna(0).astype(np.float32).values
    y = df[label_cols].fillna(0).astype(int).values

    if X.shape[0] != y.shape[0]:
        raise ValueError("X 和 y 行数不一致，请检查文件。")

    return X, y, feature_cols, label_cols, df

# =========================
# 3) 指标（macro）
# =========================
def multilabel_macro_metrics(y_true, y_prob, thresh=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= thresh).astype(int)

    L = y_true.shape[1]
    accs, precs, recs, specs = [], [], [], []
    aurocs, auprcs = [], []

    for k in range(L):
        yt = y_true[:, k]
        yp = y_pred[:, k]
        ys = y_prob[:, k]

        tp = int(np.sum((yt == 1) & (yp == 1)))
        tn = int(np.sum((yt == 0) & (yp == 0)))
        fp = int(np.sum((yt == 0) & (yp == 1)))
        fn = int(np.sum((yt == 1) & (yp == 0)))
        n = len(yt)

        accs.append((tp + tn) / n if n else np.nan)
        precs.append(tp / (tp + fp) if (tp + fp) else np.nan)
        recs.append(tp / (tp + fn) if (tp + fn) else np.nan)
        specs.append(tn / (tn + fp) if (tn + fp) else np.nan)

        if len(np.unique(yt)) == 2:
            aurocs.append(roc_auc_score(yt, ys))
            auprcs.append(average_precision_score(yt, ys))
        else:
            aurocs.append(np.nan)
            auprcs.append(np.nan)

    def nanmean(x):
        return float(np.nanmean(np.asarray(x, dtype=float)))

    return {
        "Accuracy_macro": nanmean(accs),
        "Precision_macro": nanmean(precs),
        "Recall_macro": nanmean(recs),
        "Specificity_macro": nanmean(specs),
        "AUROC_macro": nanmean(aurocs),
        "AUPRC_macro": nanmean(auprcs),
        "AUROC_valid_labels": int(np.sum(~np.isnan(aurocs))),
        "AUPRC_valid_labels": int(np.sum(~np.isnan(auprcs))),
        "n_labels": int(L),
    }

def extract_positive_proba(p, n_labels: int):
    if isinstance(p, list):
        return np.vstack([pi[:, 1] for pi in p]).T.astype(np.float32)

    p = np.asarray(p)
    if p.ndim == 3 and p.shape[-1] == 2:
        return p[:, :, 1].astype(np.float32)
    if p.ndim == 2:
        if p.shape[1] == 2 and n_labels == 1:
            return p[:, 1:2].astype(np.float32)
        return p.astype(np.float32)
    raise ValueError(f"无法解析 predict_proba 输出形状: {p.shape}")

# =========================
# 4) 近似分层 5 折：用 label cardinality 分层
# =========================
def make_stratify_target(y: np.ndarray, n_bins: int = 10):
    card = y.sum(axis=1).astype(int)
    uniq = np.unique(card)
    if len(uniq) <= 15:
        return card
    r = pd.Series(card).rank(method="average").values
    bins = pd.qcut(r, q=min(n_bins, len(np.unique(r))), labels=False, duplicates="drop")
    return np.asarray(bins, dtype=int)

def build_folds(X, y, n_splits=5, seed=42):
    strat_y = make_stratify_target(y)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(skf.split(X, strat_y))

# =========================
# 5) 5 折 CV 评估
# =========================
def cv_eval_one_paramset(X, y, folds, params, thresh=0.5):
    n_labels = y.shape[1]
    fold_metrics = []

    for fold_id, (tr_idx, va_idx) in enumerate(folds, start=1):
        X_tr, X_va = X[tr_idx], X[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        clf = xgb.XGBClassifier(**params)
        clf.fit(X_tr, y_tr)

        p = clf.predict_proba(X_va)
        y_prob = extract_positive_proba(p, n_labels=n_labels)

        m = multilabel_macro_metrics(y_va, y_prob, thresh=thresh)
        fold_metrics.append(m)

    keys = list(fold_metrics[0].keys())
    mean_metrics = {}
    for k in keys:
        vals = [fm[k] for fm in fold_metrics]
        if isinstance(vals[0], float):
            mean_metrics[k] = float(np.nanmean(vals))
        else:
            mean_metrics[k] = vals[-1]
    return mean_metrics

# =========================
# 6) Optuna 超参空间
# =========================
def build_trial_params(trial: optuna.Trial):
    params = dict(BASE_XGB_PARAMS)
    params.update({
        "n_estimators": trial.suggest_int("n_estimators", 300, 1200),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 0.5, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 5.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
    })
    return params

# =========================
# 7) 主流程：读Morgan特征 -> 固定folds -> Optuna(30) -> 输出+保存
# =========================
def main():
    check_xgb_version()

    feature_file = find_feature_file(FEATURE_FILE_GLOB)
    print("[INFO] Using transformed Morgan feature file:", feature_file)

    X, y, feat_cols, label_cols, df_all = load_Xy_from_transformed_morgan(feature_file)
    print(f"[INFO] X shape={X.shape} (features={len(feat_cols)}) | y shape={y.shape} (labels={len(label_cols)})")

    folds = build_folds(X, y, n_splits=N_SPLITS, seed=RANDOM_SEED)
    print(f"[INFO] Prepared fixed {N_SPLITS}-fold splits for all trials.")

    def objective(trial: optuna.Trial):
        params = build_trial_params(trial)
        mean_metrics = cv_eval_one_paramset(X, y, folds, params, thresh=THRESH)

        score = mean_metrics["AUPRC_macro"]  # PRAUC
        print(
            f"\n[TRIAL {trial.number:02d}] score(AUPRC_macro)={score:.6f} | "
            f"AUROC={mean_metrics['AUROC_macro']:.6f} | "
            f"Acc={mean_metrics['Accuracy_macro']:.6f} | "
            f"P={mean_metrics['Precision_macro']:.6f} | "
            f"R={mean_metrics['Recall_macro']:.6f} | "
            f"Spec={mean_metrics['Specificity_macro']:.6f}"
        )

        for k, v in mean_metrics.items():
            trial.set_user_attr(k, v)

        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

    print("\n================ BEST =================")
    print("Best trial:", study.best_trial.number)
    print("Best AUPRC_macro:", study.best_value)
    print("Best params:")
    print(json.dumps(study.best_params, indent=2, ensure_ascii=False))

    rows = []
    for t in study.trials:
        if t.value is None:
            continue
        row = {"trial": t.number, "AUPRC_macro": t.value, **t.params}
        row.update({k: v for k, v in t.user_attrs.items() if k.endswith("_macro")})
        rows.append(row)

    res_df = pd.DataFrame(rows).sort_values("AUPRC_macro", ascending=False)
    out_csv = "./optuna_singlemodel_MORGAN_30trials_5fold_results.csv"
    res_df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print("[SAVED]", out_csv)

if __name__ == "__main__":
    main()

[INFO] Using transformed Morgan feature file: ./Malodors_Rule&FG&Morgan&StructKG_features.xlsx


[I 2026-01-23 16:36:21,358] A new study created in memory with name: no-name-dc738a28-5bc8-4ffe-8c42-78a507b6b930


[INFO] X shape=(4952, 2627) (features=2627) | y shape=(4952, 138) (labels=138)
[INFO] Prepared fixed 5-fold splits for all trials.


  0%|          | 0/30 [00:00<?, ?it/s]


[TRIAL 00] score(AUPRC_macro)=0.325311 | AUROC=0.875800 | Acc=0.972269 | P=0.471021 | R=0.217710 | Spec=0.991173
[I 2026-01-23 22:53:54,837] Trial 0 finished with value: 0.3253107636382795 and parameters: {'n_estimators': 637, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 0.7978542347074177, 'reg_lambda': 0.0017775399007348214, 'reg_alpha': 0.3426417745118369, 'gamma': 3.005575058716044}. Best is trial 0 with value: 0.3253107636382795.
